## 문서 파싱 실습

`samples/` 의 문서를 **전부** 파싱해 본다. 한 건을 뜯어보는 것과 30건을 한꺼번에 돌리는 것은 보이는 게 다르다.

확인할 것
1. 파서가 전체 재료에서 버티는가
2. 같은 문서를 docx 로 읽을 때와 pdf 로 읽을 때 무엇이 달라지는가
3. 다음 단계(청킹)에 넘길 조각이 어떤 크기인가
4. 지원하지 않는 형식을 넣으면 어떻게 되는가

In [1]:
import os
import pathlib
import sys

# 노트북이 어느 위치에서 실행되든 backend/app 이 들어 있는 폴더를 찾아 루트로 삼는다
# - 폴더 이름(day01)에 기대지 않으므로 다른 날짜 노트북에 복사해도 그대로 쓸 수 있다
here = pathlib.Path.cwd().resolve()
candidates = [here, *here.parents, here / "hanwha-agent"]
ROOT = next((p for p in candidates if (p / "backend" / "app").is_dir()), None)
if ROOT is None:
    raise RuntimeError(f"hanwha-agent 루트를 찾지 못했습니다. 현재 위치: {here}")

os.chdir(ROOT)
BACKEND = str(ROOT / "backend")
if BACKEND not in sys.path:
    sys.path.insert(0, BACKEND)

SANDBOX = ROOT / "sandbox" / "w5" / "day01"
SAMPLES = SANDBOX / "samples"

print("프로젝트 루트 :", ROOT)
print("실습 재료     :", SAMPLES.relative_to(ROOT))

프로젝트 루트 : C:\workspace\hanwha-agent
실습 재료     : sandbox\w5\day01\samples


### 1. 전부 파싱해 본다

확장자를 보고 파서를 고른다. 지금 우리가 가진 파서는 둘뿐이다.

In [2]:
import logging
from app.rag.local_parsers import parse_docx, parse_pdf

# 스캔본 안내 로그가 표를 가리므로 이 셀에서만 잠시 내린다
logging.disable(logging.INFO)

PARSERS = {".pdf": parse_pdf, ".docx": parse_docx}

결과 = []
for path in sorted(SAMPLES.rglob("*")):
    if not path.is_file():
        continue
    parser = PARSERS.get(path.suffix.lower())
    if parser is None:
        continue
    try:
        doc = parser(path)
        결과.append({
            "이름": path.name,
            "형식": path.suffix.lower().lstrip("."),
            "블록": len(doc.blocks),
            "표": doc.table_count,
            "쪽": doc.page_count,
            "글자": sum(len(b.text) for b in doc.blocks),
            "오류": "",
        })
    except Exception as exc:
        결과.append({"이름": path.name, "형식": path.suffix.lower().lstrip("."),
                     "블록": 0, "표": 0, "쪽": 0, "글자": 0,
                     "오류": type(exc).__name__})

logging.disable(logging.NOTSET)

print(f"{'파일':<44}{'블록':>6}{'표':>4}{'쪽':>4}{'글자':>8}")
print("─" * 66)
for r in 결과:
    print(f"{r['이름'][:42]:<44}{r['블록']:>6}{r['표']:>4}{r['쪽']:>4}{r['글자']:>8}  {r['오류']}")

실패 = [r for r in 결과 if r["오류"]]
print("─" * 66)
print(f"{len(결과)}개 파일 · 실패 {len(실패)}건")

파일                                              블록   표   쪽      글자
──────────────────────────────────────────────────────────────────
sample_scanned.pdf                               0   0   1       0  
DOC-FI-009_법인카드_사용_지침_v1.4.docx                 81   2   1    2068  
DOC-FI-009_법인카드_사용_지침_v1.4.pdf                   6   0   6    1967  
DOC-HR-002_복무_규정_v3.1.docx                     114   3   1    3346  
DOC-HR-002_복무_규정_v3.1.pdf                        8   0   8    3200  
DOC-HR-014_국내출장_여비_규정_v1.0.docx                 60   2   1    1206  
DOC-HR-014_국내출장_여비_규정_v1.0.pdf                   6   0   6    1140  
DOC-HR-014_국내출장_여비_규정_v1.1.docx                 96   3   1    2468  
DOC-HR-014_국내출장_여비_규정_v1.1.pdf                   7   0   7    2387  
DOC-HR-014_국내출장_여비_규정_v2.0.docx                233   7   1    8273  
DOC-HR-014_국내출장_여비_규정_v2.0.pdf                  16   0  16    7801  
DOC-HR-021_재택근무_운영_지침_v1.2.docx                 95   2   1    2485  
DOC-HR-021_재택근무_운영_지침_v1.2.pdf        

### 2. 같은 문서, 두 형식

재료는 같은 규정을 docx 와 pdf 로 각각 담고 있다. 짝지어 놓고 보면 형식이 무엇을 남기고 무엇을 버리는지 드러난다.

In [3]:
# 확장자를 뗀 이름이 같은 것끼리 묶는다
짝 = {}
for r in 결과:
    if r["오류"]:
        continue
    key = r["이름"].rsplit(".", 1)[0]
    짝.setdefault(key, {})[r["형식"]] = r

완전한짝 = {k: v for k, v in 짝.items() if "docx" in v and "pdf" in v}

print(f"{'문서':<34}{'docx 블록':>10}{'pdf 블록':>10}{'배수':>7}{'docx 표':>8}{'pdf 표':>7}")
print("─" * 76)
docx_표합, pdf_표합 = 0, 0
for name, pair in sorted(완전한짝.items()):
    d, p = pair["docx"], pair["pdf"]
    docx_표합 += d["표"]
    pdf_표합 += p["표"]
    print(f"{name[:32]:<34}{d['블록']:>10}{p['블록']:>10}{d['블록'] / p['블록']:>6.1f}배{d['표']:>8}{p['표']:>7}")
print("─" * 76)
print(f"{'합계':<34}{'':>10}{'':>10}{'':>7}{docx_표합:>8}{pdf_표합:>7}")

문서                                   docx 블록    pdf 블록     배수  docx 표  pdf 표
────────────────────────────────────────────────────────────────────────────
DOC-FI-009_법인카드_사용_지침_v1.4                81         6  13.5배       2      0
DOC-HR-002_복무_규정_v3.1                    114         8  14.2배       3      0
DOC-HR-014_국내출장_여비_규정_v1.0                60         6  10.0배       2      0
DOC-HR-014_국내출장_여비_규정_v1.1                96         7  13.7배       3      0
DOC-HR-014_국내출장_여비_규정_v2.0               233        16  14.6배       7      0
DOC-HR-021_재택근무_운영_지침_v1.2                95         7  13.6배       2      0
DOC-PM-005_프로젝트_관리_지침_v2.0                80         7  11.4배       3      0
DOC-PU-007_구매·계약_규정_v4.0                 119         9  13.2배       4      0
DOC-SE-003_정보보안_지침_v2.2                  101         8  12.6배       3      0
DOC-SE-011_장비_반출입_절차_v1.0                 82         7  11.7배       3      0
────────────────────────────────────────────────────────────────────────────

표 칸을 보면 한쪽이 전부 0 이다.

`parse_pdf` 는 **표를 찾지 않는다.** PDF 에는 표라는 개념이 없고 좌표에 놓인 글자만 있어서, 선을 보는 `pdfplumber` 를 따로 써야 한다([[40_표_추출과_마크다운]]). 지금 파서는 페이지 글자만 꺼내므로 표가 문장 사이에 녹아 들어간다.

규정 문서에서 표에 들어가는 것은 금액·기한·한도다. **같은 문서인데 pdf 로 넣으면 그게 구조 없이 들어간다.**

In [4]:
# 같은 문서를 두 형식으로 읽었을 때 표가 어떻게 들어오는지
TARGET = "DOC-HR-014_국내출장_여비_규정_v2.0"

docx_doc = parse_docx(SAMPLES / f"{TARGET}.docx")
표블록 = next(b for b in docx_doc.blocks if b.kind == "표")
print("docx — 표가 마크다운 한 덩어리로 들어온다")
print("\n".join(표블록.text.splitlines()[:3]))

pdf_doc = parse_pdf(SAMPLES / f"{TARGET}.pdf")
같은쪽 = pdf_doc.blocks[1]           # 제·개정 이력이 있는 p.2
print()
print(f"pdf — 같은 내용이 {같은쪽.locator} 안에서 줄줄이 풀린다")
print(" / ".join(같은쪽.text.splitlines()[:10]))

docx — 표가 마크다운 한 덩어리로 들어온다
| 버전 | 구분 | 시행일 | 주요 내용 | 작성 | 검토 | 승인 |
|---|---|---|---|---|---|---|
| v2.0 | 개정 | 2025-07-01 | 일비 인상(25,000→30,000) · 광역시 숙박비 상한 신설 · 별표1 전면 개정 | 김지원 | 박성호 | 대표이사 |

pdf — 같은 내용이 p.2 안에서 줄줄이 풀린다
제·개정 이력 / 버전 / 구분 / 시행일 / 주요 내용 / 작성 / 검토 / 승인 / v2.0 / 개정


### 3. 청킹이 받게 될 조각

다음 단계는 이 블록들을 검색에 맞는 크기로 다시 자르는 일이다([[33_RAG_개요]] ③). 재료가 어떤 크기로 들어오는지부터 본다.

In [5]:
def 분포(사이즈: list[int]) -> str:
    사이즈 = sorted(사이즈)
    n = len(사이즈)
    중앙 = 사이즈[n // 2]
    return (f"{n:>4}개 · 평균 {sum(사이즈) // n:>4}자 · 중앙값 {중앙:>4}자 "
            f"· 최소 {사이즈[0]:>4}자 · 최대 {사이즈[-1]:>5}자")

for 형식, 파서 in (("docx", parse_docx), ("pdf", parse_pdf)):
    사이즈 = []
    for name in 완전한짝:
        doc = 파서(SAMPLES / f"{name}.{형식}")
        사이즈 += [len(b.text) for b in doc.blocks]
    print(f"{형식:<5} {분포(사이즈)}")

print()
print("평균과 중앙값이 벌어지면 크기가 고르지 않다는 뜻이다.")

docx  1061개 · 평균   29자 · 중앙값   17자 · 최소    2자 · 최대   378자
pdf     81개 · 평균  360자 · 중앙값  224자 · 최소   38자 · 최대  1070자

평균과 중앙값이 벌어지면 크기가 고르지 않다는 뜻이다.


In [6]:
# docx 블록이 실제로 어떤 것들인지 — 짧은 쪽과 긴 쪽
doc = parse_docx(SAMPLES / f"{TARGET}.docx")
짧은 = sorted(doc.blocks, key=lambda b: len(b.text))[:5]
긴 = sorted(doc.blocks, key=lambda b: len(b.text))[-3:]

print("가장 짧은 블록")
for b in 짧은:
    print(f"  {len(b.text):>3}자  [{b.kind}]  {b.text[:40]!r}")

print()
print("가장 긴 블록")
for b in 긴:
    첫줄 = b.text.splitlines()[0]
    print(f"  {len(b.text):>3}자  [{b.kind}]  {첫줄[:52]}")

가장 짧은 블록
    2자  [조항]  '일반'
    2자  [조항]  '목차'
    2자  [조항]  '부칙'
    4자  [조항]  'v2.0'
    6자  [조항]  '제1장 총칙'

가장 긴 블록
  302자  [표]  | 항 목 | 기 재 란 |
  324자  [표]  | 서류 | 필수 여부 | 제출 시점 | 보존 기간 |
  378자  [표]  | 교통수단 | 등급 | 지급 기준 | 승인 | 비고 |


짧은 쪽은 조항 번호나 한두 낱말이라 **그것만으로는 답이 안 나온다.** 긴 쪽은 표 하나가 통째라 **질문과 상관없는 행까지 딸려 온다.**

어느 쪽도 그대로 쓰기엔 맞지 않다. 파싱은 원문 구조를 보존하는 데까지이고, 크기를 맞추는 것은 청킹의 몫이다.

### 4. 지원하지 않는 형식을 넣으면

재료에는 `.hwpx` 세 건과 `_formats/` 의 다른 형식들이 섞여 있다. 위에서는 확장자로 걸러 아예 넣지 않았다. **걸르지 않으면 어떻게 되는지** 확인한다.

In [7]:
# pypdf 가 쏟아내는 경고까지 내린다 (여기서는 예외 이름만 보면 된다)
logging.disable(logging.WARNING)

후보 = [next(SAMPLES.glob("*.hwpx")), *sorted((SAMPLES / "_formats").iterdir())]

for path in 후보:
    print(path.name)
    for 이름, 파서 in (("parse_docx", parse_docx), ("parse_pdf", parse_pdf)):
        try:
            doc = 파서(path)
            print(f"   {이름:<11} 블록 {len(doc.blocks)}개")
        except Exception as exc:
            print(f"   {이름:<11} {type(exc).__name__}")
    print()

logging.disable(logging.NOTSET)

DOC-HR-014_국내출장_여비_규정_v1.1.hwpx
   parse_docx  KeyError
   parse_pdf   PdfStreamError

sample_guide.md
   parse_docx  PackageNotFoundError
   parse_pdf   PdfStreamError

sample_memo.txt
   parse_docx  PackageNotFoundError
   parse_pdf   PdfStreamError

sample_notice.pptx
   parse_docx  ValueError
   parse_pdf   PdfStreamError

sample_page.html
   parse_docx  PackageNotFoundError
   parse_pdf   PdfStreamError

sample_scanned.pdf
   parse_docx  PackageNotFoundError
   parse_pdf   블록 0개

sample_table.xlsx
   parse_docx  ValueError
   parse_pdf   PdfStreamError



예외 이름이 **무엇이 잘못됐는지 말해 주지 않는다.** `KeyError: '[Content_Types].xml'` 을 보고 "한글 파일을 워드 파서에 넣었구나"를 알아내려면 라이브러리 속을 알아야 한다.

그래서 파서에 닿기 **전에** 확장자로 가른다. [[39_문서_파싱]]의 분기 ①이 이것이다.

In [8]:
# 확장자로 먼저 가르는 자리 — 파서를 부르기 전에 판단한다
구형 = {".hwp": "HWPX", ".doc": "DOCX", ".ppt": "PPTX", ".xls": "XLSX"}
로컬 = {".pdf": parse_pdf, ".docx": parse_docx}
미구현 = {".hwpx"}


class 지원하지않는형식(Exception):
    pass


def 파서고르기(path: pathlib.Path):
    """파일 하나를 어떻게 처리할지 정한다. 읽을 수 없으면 이유를 말해준다."""
    ext = path.suffix.lower()
    if ext in 구형:
        raise 지원하지않는형식(f"{ext} 는 읽을 수 없습니다. {구형[ext]} 로 변환해 주세요.")
    if ext in 미구현:
        raise 지원하지않는형식(f"{ext} 파서는 아직 없습니다.")
    파서 = 로컬.get(ext)
    if 파서 is None:
        raise 지원하지않는형식(f"{ext} 는 다루지 않는 형식입니다.")
    return 파서


for path in [SAMPLES / "a.hwp", next(SAMPLES.glob("*.hwpx")),
             SAMPLES / "_formats" / "sample_table.xlsx",
             next(SAMPLES.glob("*.docx"))]:
    try:
        파서고르기(path)
        print(f"  통과  {path.name}")
    except 지원하지않는형식 as exc:
        print(f"  막음  {path.name:<44} {exc}")

  막음  a.hwp                                        .hwp 는 읽을 수 없습니다. HWPX 로 변환해 주세요.
  막음  DOC-HR-014_국내출장_여비_규정_v1.1.hwpx              .hwpx 파서는 아직 없습니다.
  막음  sample_table.xlsx                            .xlsx 는 다루지 않는 형식입니다.
  통과  DOC-FI-009_법인카드_사용_지침_v1.4.docx


`.hwp` 와 `.hwpx` 의 메시지가 다르다. **하나는 사용자가 할 일이 있고 하나는 우리가 할 일이 남은 것**이라, 같은 "안 됨"으로 묶으면 둘 다 고쳐지지 않는다.

이 함수는 아직 노트북 안에만 있다. `app/rag/` 로 옮길 때는 [[05_예외_계층_설계]]의 `AgentError` 아래로 넣어야 라우터가 한 번에 잡는다.

### 5. 요약 남기기

In [9]:
import json

요약 = {
    "파일": len(결과),
    "실패": len(실패),
    "짝": len(완전한짝),
    "docx_표_합계": docx_표합,
    "pdf_표_합계": pdf_표합,
    "문서별": {
        name: {"docx_블록": pair["docx"]["블록"], "pdf_블록": pair["pdf"]["블록"],
               "docx_표": pair["docx"]["표"], "쪽": pair["pdf"]["쪽"]}
        for name, pair in sorted(완전한짝.items())
    },
}

out = SANDBOX / "parse_summary.json"
out.write_text(json.dumps(요약, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"저장 : {out.relative_to(ROOT).as_posix()}  ({out.stat().st_size:,} 바이트)")
print()
print(f"문서 {요약['짝']}건 · 파일 {요약['파일']}개 · 실패 {요약['실패']}건")
print(f"표 : docx {요약['docx_표_합계']}개 / pdf {요약['pdf_표_합계']}개")

저장 : sandbox/w5/day01/parse_summary.json  (1,591 바이트)

문서 10건 · 파일 21개 · 실패 0건
표 : docx 32개 / pdf 0개


### 정리

| 확인한 것 | 결과 |
| --- | --- |
| 파서가 전체 재료에서 버티는가 | docx·pdf 21건 전부 통과, 실패 0 |
| 형식에 따라 무엇이 달라지는가 | 표를 pdf 는 하나도 못 잡는다 |
| 청킹에 넘길 조각 크기 | docx 는 너무 잘고 pdf 는 너무 굵다 |
| 지원하지 않는 형식 | 파서에 닿기 전에 확장자로 갈라야 한다 |

다음은 청킹이다. 이 두 형식이 **같은 크기의 조각**으로 나오게 만드는 것이 거기서 할 일이다.